# 🌲 Module 6: Traditional Forest Change Detection

Welcome to **Module 6** of the **Deforestation Detection from Satellite Images** project!

### 🎯 Learning Objectives
1. **Image Differencing**: Implement spectral Euclidean distance across multispectral bands.
2. **NDVI Differencing**: Compute temporal differential vegetation maps: $\Delta \text{NDVI} = \text{NDVI}_{\text{before}} - \text{NDVI}_{\text{after}}$.
3. **Threshold-Based Detection**: Explore empirical vs. optimized decision thresholds.
4. **False Positive Mitigation**: Apply morphological filtering to eliminate single-pixel noise and phenological artifacts.
5. **Benchmark Metrics**: Calculate Precision, Recall, F1-Score, IoU, and False Positive Rates to establish the baseline before Deep Learning.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
import rasterio

# Add project root to sys.path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from modules.module_06_traditional_change_detection.change_detector import (
    TraditionalChangeDetector,
    ChangeEvaluator
)

%matplotlib inline
print("✅ Module 6 traditional change detection components loaded successfully!")

## 1. Load Unseen Multi-Temporal Test Pair & Ground Truth

We evaluate traditional methods on a held-out test sample from `dataset/test/`.

In [ ]:
sample_b = "../dataset/test/before/scene_test_001.tif"
sample_a = "../dataset/test/after/scene_test_001.tif"
sample_m = "../dataset/test/masks/scene_test_001.tif"

with rasterio.open(sample_b) as src_b:
    b_bands = src_b.read()
with rasterio.open(sample_a) as src_a:
    a_bands = src_a.read()
with rasterio.open(sample_m) as src_m:
    gt_mask = src_m.read(1)

print(f"Loaded test pair with {b_bands.shape[0]} bands and shape {b_bands.shape[1:]}")
print(f"Deforested ground truth pixels: {np.sum(gt_mask == 1)} ({(np.sum(gt_mask == 1)/gt_mask.size)*100:.2f}%)")

## 2. Spectral Differencing vs. NDVI Differencing

1. **Spectral Euclidean Distance**: Measures vector magnitude shift across all 5 bands.
2. **NDVI Differencing**: Isolates photosynthetic loss: $\Delta \text{NDVI} = \text{NDVI}_{\text{before}} - \text{NDVI}_{\text{after}}$.

In [ ]:
detector = TraditionalChangeDetector()
spectral_diff = detector.compute_spectral_vector_difference(b_bands, a_bands)
delta_ndvi = detector.compute_ndvi_difference(b_bands, a_bands)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
im1 = axes[0].imshow(spectral_diff, cmap="magma")
axes[0].set_title("Spectral Euclidean Distance")
axes[0].axis("off")
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

im2 = axes[1].imshow(delta_ndvi, cmap="RdBu_r", vmin=-0.5, vmax=0.8)
axes[1].set_title("NDVI Differencing (NDVI_before - NDVI_after)", color="darkred")
axes[1].axis("off")
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)
plt.show()

## 3. Threshold Optimization Sweep

We sweep decision thresholds $\tau \in [0.05, 0.70]$ to identify the optimal cutoff maximizing F1-Score.

In [ ]:
thresholds = np.linspace(0.05, 0.70, 33)
best_thresh, best_f1, sweep_hist = detector.find_optimal_threshold(delta_ndvi, gt_mask, thresholds)

plt.figure(figsize=(9, 5))
plt.plot(sweep_hist["thresholds"], sweep_hist["f1_scores"], "r-o", linewidth=2.5, label="F1-Score (Dice)")
plt.plot(sweep_hist["thresholds"], sweep_hist["ious"], "b--s", linewidth=2, label="IoU (Jaccard)")
plt.plot(sweep_hist["thresholds"], sweep_hist["precisions"], "g-^", linewidth=1.5, alpha=0.75, label="Precision")
plt.plot(sweep_hist["thresholds"], sweep_hist["recalls"], "m-v", linewidth=1.5, alpha=0.75, label="Recall")
plt.axvline(best_thresh, color="black", linestyle=":", linewidth=2, label=f"Optimal Threshold ({best_thresh:.2f})")
plt.xlabel("Delta NDVI Threshold", fontweight="bold")
plt.ylabel("Score (0.0 to 1.0)", fontweight="bold")
plt.title("Threshold Optimization Curve", fontweight="bold")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(framealpha=0.9)
plt.show()

print(f"⭐ Optimal Threshold: {best_thresh:.3f} | Best F1-Score: {best_f1 * 100:.2f}%")

## 4. Evaluation & Error Analysis Map

Let's inspect the error breakdown:
- 🟢 **Green**: True Positives (Correctly identified deforestation)
- 🔴 **Red**: False Positives (Over-detected pixels)
- 🔵 **Blue**: False Negatives (Missed deforestation)

In [ ]:
pred_mask = detector.detect_change(delta_ndvi, threshold=best_thresh, apply_morph_cleanup=True)
metrics = ChangeEvaluator.evaluate(pred_mask, gt_mask)
ChangeEvaluator.print_metrics(metrics)

error_rgb = np.zeros((*gt_mask.shape, 3), dtype=np.float32)
error_rgb[(pred_mask == 1) & (gt_mask == 1)] = [0.15, 0.75, 0.25]  # Green
error_rgb[(pred_mask == 1) & (gt_mask == 0)] = [0.90, 0.15, 0.15]  # Red
error_rgb[(pred_mask == 0) & (gt_mask == 1)] = [0.20, 0.40, 0.90]  # Blue

plt.figure(figsize=(7, 7))
plt.imshow(error_rgb)
plt.title(f"Error Analysis Map (F1: {metrics['f1_score']*100:.1f}%, IoU: {metrics['iou']*100:.1f}%)\n[Green: TP, Red: FP, Blue: FN]", fontweight="bold")
plt.axis("off")
plt.show()